# Notebook 04.1: RAG + Context Reranking (OpenAI GPT-4.1-mini)

**Pipeline:** Query -> BM25 top-20 -> CrossEncoder Rerank top-5 -> LLM Generate
**Reranker:** `cross-encoder/ms-marco-MiniLM-L-6-v2` (lokal)
**Evaluasi:** 4 metrik RAGAS custom zero-NaN

## 1. Impor Library

In [1]:
import os, sys, json, pickle, time, re, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime

from openai import OpenAI
from rank_bm25 import BM25Okapi
from datasets import load_dataset
from sentence_transformers import CrossEncoder

warnings.filterwarnings('ignore')
print('Semua library berhasil diimpor!')
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__} | Pandas: {pd.__version__}')

C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Semua library berhasil diimpor!
Python: 3.11.9 | NumPy: 2.3.5 | Pandas: 2.3.3


## 2. Konfigurasi

In [ ]:
# ============================================================
# PATH SETUP — auto-resolve PROJECT_ROOT
# ============================================================
from pathlib import Path

_HERE = Path('.').resolve()
PROJECT_ROOT = next((p for p in [_HERE] + list(_HERE.parents) if p.name == 'Code TA'), _HERE.parent.parent.parent)
NOTEBOOKS_V2 = PROJECT_ROOT / 'notebooks'
INDEXES_DIR  = NOTEBOOKS_V2 / 'indexes'
RESULTS_DIR  = PROJECT_ROOT / 'results' / '10_bm25'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BM25_INDEX_PATH = INDEXES_DIR / 'pubmedqa_bm25.pkl'
NOTEBOOK_DIR    = INDEXES_DIR  # kompat lama: BM25_INDEX_PATH dan CHROMA_DB_PATH

print(f'PROJECT_ROOT  : {PROJECT_ROOT}')
print(f'INDEXES_DIR   : {INDEXES_DIR}')
print(f'RESULTS_DIR   : {RESULTS_DIR}')
print(f'BM25 index    : {BM25_INDEX_PATH.name}')


In [2]:
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'YOUR_OPENAI_KEY_HERE')

LLM_MODEL        = 'gpt-4.1-mini'
RERANKER_MODEL   = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
TOP_K_CANDIDATES = 20
TOP_K_RETRIEVAL  = 5

DATASET_NAME   = 'qiaojin/PubMedQA'
DATASET_SUBSET = 'pqa_labeled'
MAX_SAMPLES    = 500

TEMPERATURE = 0.0
SEED        = 42

# NOTEBOOK_DIR    = Path('.')
# BM25_INDEX_PATH = NOTEBOOK_DIR / 'pubmedqa_bm25.pkl'
# RESULTS_DIR     = Path('../results')
# RESULTS_DIR.mkdir(exist_ok=True)

CONFIG_NAME    = 'cr_openai'
PHASE1_PATH    = RESULTS_DIR / f'{CONFIG_NAME}_phase1_answers.json'

BASELINE_PHASE1_PATH = RESULTS_DIR / 'baseline_openai_phase1_answers.json'
BASELINE_PHASE2_PATH = RESULTS_DIR / 'baseline_openai_phase2_custom.json'

print('Konfigurasi:')
print(f'  LLM        : {LLM_MODEL} (via OpenAI API)')
print(f'  Reranker   : {RERANKER_MODEL} (lokal)')
print(f'  BM25 top-{TOP_K_CANDIDATES} -> Rerank top-{TOP_K_RETRIEVAL}')
print(f'  Sampel     : {MAX_SAMPLES}')
print(f'  Config     : {CONFIG_NAME}')

Konfigurasi:
  LLM        : gpt-4.1-mini (via OpenAI API)
  Reranker   : cross-encoder/ms-marco-MiniLM-L-6-v2 (lokal)
  BM25 top-20 -> Rerank top-5
  Sampel     : 500
  Config     : cr_openai


In [3]:
# ============================================================
# Setup OpenAI Client
# ============================================================

openai_client = OpenAI(api_key=OPENAI_API_KEY)

def openai_generate(prompt: str, max_tokens: int = 300, temperature: float = TEMPERATURE) -> str:
    for attempt in range(5):
        try:
            response = openai_client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
                seed=SEED,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower():
                wait = (attempt + 1) * 10
                print(f'  [Rate limit] Tunggu {wait}s... (attempt {attempt+1}/5)')
                time.sleep(wait)
            elif '500' in err or '502' in err or '503' in err:
                wait = (attempt + 1) * 5
                print(f'  [Server error] Tunggu {wait}s... (attempt {attempt+1}/5)')
                time.sleep(wait)
            else:
                print(f'  [OpenAI Error] {type(e).__name__}: {err[:100]}')
                raise
    raise RuntimeError('OpenAI API gagal setelah 5 percobaan.')

print('Testing OpenAI API...')
_test = openai_generate('Reply with exactly: OK', max_tokens=5)
print(f'Response: {_test!r} | Model: {LLM_MODEL}')
print('OpenAI client siap!')

Testing OpenAI API...
Response: 'OK' | Model: gpt-4.1-mini
OpenAI client siap!


## 3. Data Classes dan Tokenizer BM25

In [4]:
@dataclass
class Document:
    text         : str
    pubid        : str
    question     : str
    section_label: str
    answer       : str
    decision     : str

@dataclass
class RetrievalResult:
    document      : Document
    score         : float           # BM25 score (kandidat awal)
    reranker_score: float = 0.0     # CrossEncoder score (setelah rerank)


def tokenize_bm25(text: str) -> List[str]:
    """Tokenizer untuk BM25: hapus tanda baca, lowercase, split spasi."""
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower()).split()


sample_text = 'Does aspirin (75mg) reduce myocardial infarction risk?'
print(f'Tokenisasi BM25: {tokenize_bm25(sample_text)}')
print('Data classes dan tokenizer siap.')

Tokenisasi BM25: ['does', 'aspirin', '75mg', 'reduce', 'myocardial', 'infarction', 'risk']
Data classes dan tokenizer siap.


## 4. Muat Dataset dan Bangun BM25 Index

BM25 index di-share dengan notebook baseline dan QR — gunakan file `pubmedqa_bm25.pkl` yang sama.
Jika sudah ada, proses muat hanya butuh beberapa detik.

In [5]:
def load_pubmedqa(subset=DATASET_SUBSET, max_samples=MAX_SAMPLES):
    print(f'Memuat PubMedQA ({subset})...')
    dataset = load_dataset(DATASET_NAME, subset, trust_remote_code=True)
    data    = dataset['train']
    if max_samples and len(data) > max_samples:
        data = data.select(range(max_samples))
    print(f'Dimuat {len(data)} sampel')
    return data


def prepare_documents(data) -> List[Document]:
    docs = []
    for item in data:
        pubid = str(item['pubid'])
        for ctx, label in zip(item['context']['contexts'], item['context']['labels']):
            docs.append(Document(
                text=ctx.strip(), pubid=pubid,
                question=item['question'], section_label=label,
                answer=item['long_answer'], decision=item['final_decision']
            ))
    print(f'Total potongan dokumen: {len(docs)}')
    return docs


def load_or_build_bm25(data) -> Tuple[BM25Okapi, List[Document]]:
    if BM25_INDEX_PATH.exists():
        print(f'Memuat BM25 index dari {BM25_INDEX_PATH}...')
        with open(BM25_INDEX_PATH, 'rb') as f:
            saved = pickle.load(f)
        print(f'Dimuat: {len(saved["documents"])} dokumen')
        return saved['bm25'], saved['documents']
    else:
        print('Membangun BM25 index...')
        documents = prepare_documents(data)
        tokenized = [tokenize_bm25(d.text) for d in documents]
        bm25      = BM25Okapi(tokenized)
        with open(BM25_INDEX_PATH, 'wb') as f:
            pickle.dump({'bm25': bm25, 'documents': documents}, f)
        print(f'Index disimpan ke {BM25_INDEX_PATH}')
        return bm25, documents


t0 = time.time()
pubmedqa_data         = load_pubmedqa()
bm25_index, documents = load_or_build_bm25(pubmedqa_data)
print(f'Selesai dalam {time.time()-t0:.1f} detik')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Memuat PubMedQA (pqa_labeled)...


Dimuat 500 sampel
Memuat BM25 index dari pubmedqa_bm25.pkl...
Dimuat: 1706 dokumen
Selesai dalam 6.0 detik


## 5. Muat CrossEncoder Reranker

**Model:** `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Dilatih pada MS MARCO passage ranking (passage retrieval untuk QA)
- Lightweight: 6-layer MiniLM, ~22M parameter
- Input: pasangan (query, passage) → output: relevance score
- Tidak memerlukan GPU — berjalan di CPU

**Kenapa tidak bi-encoder?**
Bi-encoder encode query dan doc secara terpisah (embedding cosine similarity).
Cross-encoder membaca keduanya sekaligus → representasi interaksi lebih kaya → lebih akurat tapi lebih lambat.
Untuk reranking top-20 (bukan full corpus), cross-encoder trade-off latency-nya ok.

In [6]:
print(f'Memuat CrossEncoder: {RERANKER_MODEL}...')
print('(Download ~85MB sekali, lalu di-cache)')
t0 = time.time()
cross_encoder = CrossEncoder(RERANKER_MODEL)
print(f'CrossEncoder siap dalam {time.time()-t0:.1f} detik')

# Smoke test
_pairs = [
    ('Does aspirin prevent heart attacks?', 'Aspirin reduces platelet aggregation and is used in cardiovascular prevention.'),
    ('Does aspirin prevent heart attacks?', 'Weather patterns affect agricultural yields in tropical regions.'),
]
_scores = cross_encoder.predict(_pairs)
print(f'\nSmoke test CrossEncoder:')
print(f'  Relevan   : {_scores[0]:.4f}')
print(f'  Tidak relevan: {_scores[1]:.4f}')
assert _scores[0] > _scores[1], 'CrossEncoder gagal membedakan relevan vs tidak!'
print('Reranker berfungsi dengan benar.')

Memuat CrossEncoder: cross-encoder/ms-marco-MiniLM-L-6-v2...
(Download ~85MB sekali, lalu di-cache)


Loading weights: 100%|███████████████████████| 105/105 [00:00<00:00, 191.33it/s, Materializing param=classifier.weight]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CrossEncoder siap dalam 6.1 detik

Smoke test CrossEncoder:
  Relevan   : 4.9414
  Tidak relevan: -11.1675
Reranker berfungsi dengan benar.


## 6. Fungsi Retrieval dengan Context Reranking (BM25 → CrossEncoder)

In [7]:
def retrieve_with_reranking(
    query: str,
    k_candidates: int = TOP_K_CANDIDATES,
    k_final: int = TOP_K_RETRIEVAL
) -> List[RetrievalResult]:
    """
    Retrieval dua tahap:
      1. BM25: ambil top-k_candidates (20) dokumen kandidat
      2. CrossEncoder: rerank kandidat, ambil top-k_final (5)

    Returns list RetrievalResult dengan atribut:
      .score          = BM25 score asli
      .reranker_score = CrossEncoder score setelah reranking
    """
    # Tahap 1: BM25 retrieval
    tokens    = tokenize_bm25(query)
    scores    = bm25_index.get_scores(tokens)
    top_cands = np.argsort(scores)[::-1][:k_candidates]
    candidates = [
        RetrievalResult(document=documents[i], score=float(scores[i]))
        for i in top_cands
    ]

    # Tahap 2: CrossEncoder reranking
    pairs = [(query, r.document.text) for r in candidates]
    reranker_scores = cross_encoder.predict(pairs)

    for r, rs in zip(candidates, reranker_scores):
        r.reranker_score = float(rs)

    # Sort by reranker score, ambil top-k_final
    reranked = sorted(candidates, key=lambda r: r.reranker_score, reverse=True)
    return reranked[:k_final]


# Test retrieval
test_q = 'Does aspirin reduce the risk of myocardial infarction?'
test_r = retrieve_with_reranking(test_q)
print(f'Query: {test_q}')
print(f'\nTop-{TOP_K_RETRIEVAL} dokumen (BM25 top-{TOP_K_CANDIDATES} → CrossEncoder):')  
for i, r in enumerate(test_r, 1):
    print(f'  [{i}] BM25={r.score:.2f} | Reranker={r.reranker_score:.4f} | {r.document.section_label} | {r.document.text[:80]}...')

Query: Does aspirin reduce the risk of myocardial infarction?

Top-5 dokumen (BM25 top-20 → CrossEncoder):
  [1] BM25=12.18 | Reranker=-2.5403 | BACKGROUND AND PURPOSE | In primary and secondary prevention trials, statins have been shown to reduce th...
  [2] BM25=16.24 | Reranker=-3.1160 | OBJECTIVE | Myocardial damage that is associated with percutaneous coronary intervention (PC...
  [3] BM25=23.49 | Reranker=-5.1495 | DESIGN | Within a prospective, population-based cohort study individuals without history ...
  [4] BM25=15.69 | Reranker=-5.3794 | BACKGROUND | It has recently been shown that non-high density lipoprotein cholesterol (non-HD...
  [5] BM25=19.31 | Reranker=-5.6010 | METHODS | Of the 9681 women and 8888 men who attended risk assessment from 1967-1991, with...


## 7. Prompt Generasi dan Fungsi Generate

Prompt **identik** dengan baseline — isolasi variabel: hanya retrieval yang berubah, bukan generation.

In [8]:
GENERATION_PROMPT = (
    'You are a medical research assistant. '
    'Answer a biomedical yes/no/maybe question based solely on the provided scientific abstracts.\n\n'
    'Context from medical literature:\n{context}\n\n'
    'Question: {question}\n\n'
    'Instructions:\n'
    '- Carefully read the context and assess whether it supports or refutes the question.\n'
    '- Provide a brief explanation (2-3 sentences) using ONLY the information above.\n'
    '- End your response with EXACTLY ONE of these words on its own line: yes, no, or maybe.\n'
    '  - yes   : the evidence supports the hypothesis, even if not perfectly conclusive\n'
    '  - no    : the evidence refutes or does not support the hypothesis\n'
    '  - maybe : ONLY if the evidence is directly contradictory (some findings say yes,\n'
    '            others say no), or if the context contains no relevant information at all\n'
    '- IMPORTANT: If the evidence leans in one direction, even partially, choose yes or no.\n'
    '  Do NOT use maybe simply because the evidence is limited or not 100%% certain.\n\n'
    'Answer:'
)

def generate_answer(query: str, retrieved: List[RetrievalResult]) -> str:
    context = '\n\n'.join(
        f'[{i}] ({r.document.section_label}): {r.document.text}'
        for i, r in enumerate(retrieved, 1)
    )
    return openai_generate(
        GENERATION_PROMPT.format(context=context, question=query),
        max_tokens=300, temperature=TEMPERATURE
    )

test_ans = generate_answer(test_q, test_r)
print('Output generation:')
print('-' * 60)
print(test_ans)

Output generation:
------------------------------------------------------------
The provided abstracts discuss the effects of statins on stroke risk, myocardial damage related to PCI, and lipid predictors of cardiovascular risk, but none mention aspirin or its impact on myocardial infarction risk. Therefore, there is no information here to support or refute the role of aspirin in reducing myocardial infarction risk.

maybe


## 8. Ekstraksi Label yes/no/maybe

In [9]:
def extract_label(answer: str) -> str:
    """
    Ekstrak prediksi yes/no/maybe dari teks jawaban.
    Strategi (berurutan hingga ditemukan):
      1. Kata standalone di 3 baris terakhir (non-kosong)
      2. Kata standalone di seluruh teks
      3. Default ke 'maybe'
    """
    lines = [l.strip().lower() for l in answer.split('\n') if l.strip()]
    for line in reversed(lines[-3:]):
        word = re.sub(r'[^a-z]', '', line)
        if word in ('yes', 'no', 'maybe'):
            return word
    for label in ('yes', 'no', 'maybe'):
        if re.search(r'\b' + label + r'\b', answer.lower()):
            return label
    return 'maybe'


cases = [
    ('Strong evidence.\nyes',    'yes'),
    ('No effect found.\nno',     'no'),
    ('Mixed results.\nmaybe',    'maybe'),
    ('Verdict: yes.',             'yes'),
    ('Totally unclear.',          'maybe'),
]
print('Unit test extract_label:')
all_ok = True
for txt, exp in cases:
    pred = extract_label(txt)
    ok   = pred == exp
    all_ok = all_ok and ok
    print(f'  [{"PASS" if ok else "FAIL"}] pred={pred!r} expected={exp!r}')
print(f'\nSemua lulus: {all_ok}')
print(f'Label dari test answer: {extract_label(test_ans)!r}')

Unit test extract_label:
  [PASS] pred='yes' expected='yes'
  [PASS] pred='no' expected='no'
  [PASS] pred='maybe' expected='maybe'
  [PASS] pred='yes' expected='yes'
  [PASS] pred='maybe' expected='maybe'

Semua lulus: True
Label dari test answer: 'maybe'


---
## Custom Evaluator 4 Metrik (Zero-NaN)

faithfulness, context_recall, answer_relevancy, context_precision — semua dijamin 0.0-1.0.

In [10]:
# ============================================================
# Custom Zero-NaN Evaluator -- 4 Metrik RAGAS (via OpenAI)
# Selalu return 0.0-1.0, TIDAK PERNAH NaN
# ============================================================

def _split_sentences(text: str) -> List[str]:
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in parts if len(s.strip()) >= 15]

def _llm_yes_no(prompt: str) -> bool:
    try:
        resp = openai_generate(prompt, max_tokens=10, temperature=0.0)
        return 'yes' in resp.lower()[:15]
    except Exception:
        return False

def compute_faithfulness(answer: str, contexts: List[str]) -> float:
    sentences = _split_sentences(answer)
    if not sentences: return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = ('Context:\n{ctx}\n\nStatement: {sent}\n\n'
        'Is this statement directly supported by the context above? Answer with only "yes" or "no".')
    supported = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s)))
    return supported / len(sentences)

def compute_context_recall(reference: str, contexts: List[str]) -> float:
    sentences = _split_sentences(reference)
    if not sentences: return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = ('Context:\n{ctx}\n\nStatement: {sent}\n\n'
        'Is this statement supported by the context above? Answer with only "yes" or "no".')
    covered = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s)))
    return covered / len(sentences)

def compute_answer_relevancy(question: str, answer: str) -> float:
    sentences = _split_sentences(answer)
    if not sentences: return 0.0
    prompt_tmpl = ('Question: {question}\n\nStatement: {sent}\n\n'
        'Is this statement relevant to answering the question above? Answer with only "yes" or "no".')
    relevant = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(question=question, sent=s)))
    return relevant / len(sentences)

def compute_context_precision(question: str, contexts: List[str], reference: str) -> float:
    if not contexts: return 0.0
    prompt_tmpl = ('Question: {question}\n\nGround truth answer: {reference}\n\n'
        'Retrieved context: {ctx}\n\nDoes this context contain information useful for correctly '
        'answering the question based on the ground truth? Answer with only "yes" or "no".')
    relevance = []
    for ctx in contexts:
        is_rel = _llm_yes_no(prompt_tmpl.format(question=question, reference=reference[:300], ctx=ctx[:400]))
        relevance.append(1 if is_rel else 0)
    total_relevant = sum(relevance)
    if total_relevant == 0: return 0.0
    precision_sum, relevant_count = 0.0, 0
    for k, rel in enumerate(relevance):
        if rel:
            relevant_count += 1
            precision_sum += relevant_count / (k + 1)
    return precision_sum / total_relevant

def evaluate_custom(question, answer, contexts, reference):
    return {
        'faithfulness': compute_faithfulness(answer, contexts),
        'context_recall': compute_context_recall(reference, contexts),
        'answer_relevancy': compute_answer_relevancy(question, answer),
        'context_precision': compute_context_precision(question, contexts, reference),
    }

_ctx = ['Aspirin reduces blood clotting and is used for heart attack prevention.']
_ans = 'Aspirin helps prevent heart attacks. It works by reducing clotting.'
_ref = 'Aspirin is used for heart attack prevention by reducing blood clotting.'
_r = evaluate_custom('Does aspirin prevent heart attacks?', _ans, _ctx, _ref)
print(f'Smoke test (4 metrik): faith={_r["faithfulness"]:.2f} cr={_r["context_recall"]:.2f} ar={_r["answer_relevancy"]:.2f} cp={_r["context_precision"]:.2f}')
print('Zero-NaN evaluator siap (4 metrik via OpenAI).')

Smoke test (4 metrik): faith=1.00 cr=1.00 ar=1.00 cp=1.00
Zero-NaN evaluator siap (4 metrik via OpenAI).


---
## DEMO: Uji Coba 5 Sampel

Verifikasi pipeline CR berjalan dengan benar.
Tampilkan BM25 kandidat vs hasil reranking untuk tiap sampel.

> Estimasi: ~2–3 menit

In [11]:
DEMO_SIZE    = 5
demo_results = []

print(f'DEMO: {DEMO_SIZE} sampel pertama (RAG + Context Reranking)')
print('=' * 70)

for i in range(DEMO_SIZE):
    s          = pubmedqa_data[i]
    q, gt, ref = s['question'], s['final_decision'], s['long_answer']

    # Retrieve: BM25 top-20 → CrossEncoder → top-5
    retrieved = retrieve_with_reranking(q)

    # Generate
    answer    = generate_answer(q, retrieved)
    predicted = extract_label(answer)

    demo_results.append({
        'idx': i, 'question': q, 'ground_truth': gt,
        'predicted_label': predicted, 'is_correct': predicted == gt,
        'answer': answer,
        'bm25_scores'    : [r.score for r in retrieved],
        'reranker_scores': [r.reranker_score for r in retrieved],
    })

    status = 'BENAR' if predicted == gt else 'SALAH'
    print(f'\n[{i}] {q[:70]}')
    print(f'     GT={gt} | Pred={predicted} | {status}')
    print(f'     Reranker scores: {[f"{r.reranker_score:.3f}" for r in retrieved]}')
    print(f'     Jawaban: {answer[:100]}...')

demo_acc = sum(r['is_correct'] for r in demo_results) / DEMO_SIZE
print(f'\nDemo accuracy: {demo_acc:.0%} ({sum(r["is_correct"] for r in demo_results)}/{DEMO_SIZE})')
print('Pipeline CR siap untuk Fase 1.')

---
## Fase 1: Generate Semua Jawaban (500 Sampel)

BM25 top-20 → CrossEncoder rerank → top-5 → LLM generate.
Disimpan inkremental setiap 10 sampel — resume-able jika terputus.

Field tambahan vs baseline: `reranker_scores`, `bm25_scores_candidates` (skor BM25 top-20 sebelum rerank)

> Estimasi: ~4–6 jam (CrossEncoder inference di CPU lebih lambat dari BM25 saja)

In [11]:
if PHASE1_PATH.exists():
    with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
        phase1_results = json.load(f)['results']
    start_from = len(phase1_results)
    print(f'Resume Fase 1: {start_from}/{MAX_SAMPLES} sudah selesai.')
else:
    phase1_results, start_from = [], 0
    print(f'Memulai Fase 1: {MAX_SAMPLES} sampel (BM25 top-{TOP_K_CANDIDATES} → CR → top-{TOP_K_RETRIEVAL}).')

if start_from < MAX_SAMPLES:
    print(f'Memproses {MAX_SAMPLES - start_from} sampel tersisa...\n')
    t_start = time.time()
    for i in range(start_from, MAX_SAMPLES):
        s          = pubmedqa_data[i]
        q, gt, ref = s['question'], s['final_decision'], s['long_answer']

        retrieved  = retrieve_with_reranking(q)
        answer     = generate_answer(q, retrieved)
        predicted  = extract_label(answer)

        phase1_results.append({
            'idx'             : i,
            'pubid'           : str(s['pubid']),
            'question'        : q,
            'ground_truth'    : gt,
            'predicted_label' : predicted,
            'is_correct'      : predicted == gt,
            'answer'          : answer,
            'contexts'        : [r.document.text for r in retrieved],
            'reference'       : ref,
            'retrieval_scores': [r.score for r in retrieved],
            'reranker_scores' : [r.reranker_score for r in retrieved],
        })

        if (i + 1) % 10 == 0 or i == MAX_SAMPLES - 1:
            with open(PHASE1_PATH, 'w', encoding='utf-8') as f:
                json.dump({'config': CONFIG_NAME,
                           'timestamp': datetime.now().isoformat(),
                           'max_samples': MAX_SAMPLES, 'completed': i+1,
                           'results': phase1_results}, f, indent=2, ensure_ascii=False)
            done = i + 1
            acc  = sum(r['is_correct'] for r in phase1_results) / done
            eta  = (time.time()-t_start) / done * (MAX_SAMPLES-done) / 60
            print(f'  [{done:3d}/{MAX_SAMPLES}] Akurasi: {acc:.1%} | pred={predicted}, gt={gt} | ETA {eta:.1f} mnt')
    print(f'\nFase 1 selesai! Disimpan ke {PHASE1_PATH}')
else:
    print(f'Fase 1 sudah selesai ({MAX_SAMPLES} sampel).')

Memulai Fase 1: 500 sampel (BM25 top-20 → CR → top-5).
Memproses 500 sampel tersisa...

  [ 10/500] Akurasi: 40.0% | pred=yes, gt=yes | ETA 25.0 mnt
  [ 20/500] Akurasi: 60.0% | pred=yes, gt=yes | ETA 23.6 mnt
  [ 30/500] Akurasi: 66.7% | pred=yes, gt=yes | ETA 22.9 mnt
  [ 40/500] Akurasi: 62.5% | pred=no, gt=no | ETA 21.6 mnt
  [ 50/500] Akurasi: 64.0% | pred=no, gt=no | ETA 21.0 mnt
  [ 60/500] Akurasi: 60.0% | pred=yes, gt=yes | ETA 20.4 mnt
  [ 70/500] Akurasi: 64.3% | pred=yes, gt=yes | ETA 19.5 mnt
  [ 80/500] Akurasi: 62.5% | pred=yes, gt=yes | ETA 19.1 mnt
  [ 90/500] Akurasi: 62.2% | pred=no, gt=maybe | ETA 18.6 mnt
  [100/500] Akurasi: 64.0% | pred=yes, gt=yes | ETA 18.3 mnt
  [110/500] Akurasi: 64.5% | pred=yes, gt=yes | ETA 18.1 mnt
  [120/500] Akurasi: 63.3% | pred=yes, gt=yes | ETA 17.7 mnt
  [130/500] Akurasi: 61.5% | pred=yes, gt=maybe | ETA 17.2 mnt
  [140/500] Akurasi: 62.1% | pred=yes, gt=yes | ETA 16.8 mnt
  [150/500] Akurasi: 61.3% | pred=yes, gt=no | ETA 16.4 mnt

### Analisis Fase 1 (Label Accuracy)

In [12]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']
n         = len(results_p1)
n_correct = sum(r['is_correct'] for r in results_p1)
gts       = [r['ground_truth']    for r in results_p1]
preds     = [r['predicted_label'] for r in results_p1]

print(f'ANALISIS FASE 1 -- {n} sampel (RAG + Context Reranking)')
print('=' * 55)
print(f'Label Accuracy    : {n_correct}/{n} = {n_correct/n:.1%}')
print(f'Hallucination Rate: {(n-n_correct)/n:.1%}\n')
print(f'  {"Label":<8} | {"Ground Truth":>12} | {"Prediksi":>12}')
print(f'  {"-"*8}-+{"-"*14}-+{"-"*12}')
for lbl in ['yes','no','maybe']:
    g, p = gts.count(lbl), preds.count(lbl)
    print(f'  {lbl:<8} | {g:>10} ({g/n:.0%}) | {p:>10} ({p/n:.0%})')

print('\nConfusion Matrix (baris=GT, kolom=Prediksi):')
lbls = ['yes','no','maybe']
print('  ' + f'{"GT/Pred":>8}' + ''.join(f'{l:>8}' for l in lbls))
for gt_l in lbls:
    row = f'  {gt_l:>8}'
    for pr_l in lbls:
        cnt = sum(1 for r in results_p1 if r['ground_truth']==gt_l and r['predicted_label']==pr_l)
        row += f'{cnt:>8}'
    print(row)

# Statistik reranker score
all_rr_scores = [s for r in results_p1 for s in r.get('reranker_scores', [])]
all_bm25_scores = [s for r in results_p1 for s in r.get('retrieval_scores', [])]
if all_rr_scores:
    print(f'\nStatistik Reranker Score (top-{TOP_K_RETRIEVAL} per sampel):')
    print(f'  Mean  : {np.mean(all_rr_scores):.4f}')
    print(f'  Median: {np.median(all_rr_scores):.4f}')
    print(f'  Min   : {np.min(all_rr_scores):.4f}')
    print(f'  Max   : {np.max(all_rr_scores):.4f}')

ANALISIS FASE 1 -- 500 sampel (RAG + Context Reranking)
Label Accuracy    : 342/500 = 68.4%
Hallucination Rate: 31.6%

  Label    | Ground Truth |     Prediksi
  ---------+---------------+------------
  yes      |        275 (55%) |        326 (65%)
  no       |        159 (32%) |        147 (29%)
  maybe    |         66 (13%) |         27 (5%)

Confusion Matrix (baris=GT, kolom=Prediksi):
   GT/Pred     yes      no   maybe
       yes     238      26      11
        no      45     101      13
     maybe      43      20       3

Statistik Reranker Score (top-5 per sampel):
  Mean  : -2.4110
  Median: -3.5116
  Min   : -11.3816
  Max   : 10.8021


---
## Phase 2: Evaluasi Custom 4 Metrik (500 Sampel)

Estimasi: ~30-60 menit. Resume otomatis.

In [11]:
MAX_CUSTOM_SAMPLES = 500
PHASE2_CUSTOM_PATH = RESULTS_DIR / f'{CONFIG_NAME}_phase2_custom.json'

with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    p1_custom = json.load(f)['results'][:MAX_CUSTOM_SAMPLES]

if PHASE2_CUSTOM_PATH.exists():
    with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
        p2_custom = json.load(f)['results']
    done_custom = {r['idx'] for r in p2_custom}
    print(f'Resume: {len(done_custom)}/{MAX_CUSTOM_SAMPLES} selesai.')
else:
    p2_custom, done_custom = [], set()
    print(f'Mulai: {MAX_CUSTOM_SAMPLES} sampel (custom zero-NaN, 4 metrik).')

remaining = [r for r in p1_custom if r['idx'] not in done_custom]
print(f'Sisa: {len(remaining)} sampel\n')

t0 = time.time()
for i, r in enumerate(remaining):
    scores = evaluate_custom(r['question'], r['answer'], r['contexts'], r['reference'])
    p2_custom.append({'idx': r['idx'], 'ground_truth': r['ground_truth'],
        'predicted_label': r['predicted_label'], 'is_correct': r['is_correct'], **scores})
    if (i+1) % 5 == 0 or i == len(remaining)-1:
        with open(PHASE2_CUSTOM_PATH, 'w', encoding='utf-8') as f:
            json.dump({'config': CONFIG_NAME, 'llm_model': LLM_MODEL,
                'timestamp': datetime.now().isoformat(), 'max_samples': MAX_CUSTOM_SAMPLES,
                'metrics': ['faithfulness','context_recall','answer_relevancy','context_precision'],
                'evaluator': 'custom_zero_nan_4metrics', 'results': p2_custom
            }, f, indent=2, ensure_ascii=False)
        done, total = i+1, len(remaining)
        eta = (time.time()-t0)/done*(total-done)/60 if done < total else 0
        avg_f  = sum(x['faithfulness'] for x in p2_custom)/len(p2_custom)
        avg_cr = sum(x['context_recall'] for x in p2_custom)/len(p2_custom)
        avg_ar = sum(x['answer_relevancy'] for x in p2_custom)/len(p2_custom)
        avg_cp = sum(x['context_precision'] for x in p2_custom)/len(p2_custom)
        print(f'  [{done:3d}/{total}] f={avg_f:.3f} cr={avg_cr:.3f} ar={avg_ar:.3f} cp={avg_cp:.3f} | ETA {eta:.1f}m')

n = len(p2_custom)
acc = sum(r['is_correct'] for r in p2_custom)/n
avg_f  = sum(r['faithfulness'] for r in p2_custom)/n
avg_cr = sum(r['context_recall'] for r in p2_custom)/n
avg_ar = sum(r['answer_relevancy'] for r in p2_custom)/n
avg_cp = sum(r['context_precision'] for r in p2_custom)/n
print(f'\nSelesai! {n} sampel:')
print(f'  Accuracy={acc:.1%%} | faith={avg_f:.4f} cr={avg_cr:.4f} ar={avg_ar:.4f} cp={avg_cp:.4f}')
print(f'\n  | {CONFIG_NAME} | {acc:.3f} | {1-acc:.3f} | {avg_f:.3f} | {avg_cr:.3f} | {avg_ar:.3f} | {avg_cp:.3f} |')

Mulai: 500 sampel (custom zero-NaN, 4 metrik).
Sisa: 500 sampel

  [  5/500] f=0.767 cr=0.650 ar=1.000 cp=0.900 | ETA 84.0m
  [ 10/500] f=0.883 cr=0.658 ar=1.000 cp=0.817 | ETA 79.2m
  [ 15/500] f=0.856 cr=0.739 ar=1.000 cp=0.744 | ETA 76.6m
  [ 20/500] f=0.892 cr=0.804 ar=1.000 cp=0.799 | ETA 72.8m
  [ 25/500] f=0.893 cr=0.843 ar=1.000 cp=0.816 | ETA 72.1m
  [ 30/500] f=0.900 cr=0.836 ar=1.000 cp=0.796 | ETA 71.3m
  [ 35/500] f=0.900 cr=0.860 ar=1.000 cp=0.806 | ETA 70.0m
  [ 40/500] f=0.912 cr=0.852 ar=1.000 cp=0.789 | ETA 71.8m
  [ 45/500] f=0.907 cr=0.835 ar=1.000 cp=0.764 | ETA 72.9m
  [ 50/500] f=0.917 cr=0.832 ar=0.993 cp=0.788 | ETA 72.0m
  [ 55/500] f=0.924 cr=0.829 ar=0.994 cp=0.767 | ETA 71.3m
  [ 60/500] f=0.922 cr=0.824 ar=0.994 cp=0.785 | ETA 72.4m
  [ 65/500] f=0.928 cr=0.806 ar=0.995 cp=0.779 | ETA 71.6m
  [ 70/500] f=0.926 cr=0.820 ar=0.995 cp=0.778 | ETA 71.4m
  [ 75/500] f=0.931 cr=0.821 ar=0.996 cp=0.774 | ETA 70.1m
  [ 80/500] f=0.935 cr=0.832 ar=0.996 cp=0.759 | E

ValueError: Invalid format specifier '.1%%' for object of type 'float'

---
## Ringkasan Metrik Evaluasi Context Reranking

In [12]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']
with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
    results_p2 = json.load(f)['results']
p2_lu = {r['idx']: r for r in results_p2}

df = pd.DataFrame([{
    'idx'             : r['idx'],
    'ground_truth'    : r['ground_truth'],
    'predicted_label' : r['predicted_label'],
    'is_correct'      : r['is_correct'],
    'faithfulness'    : p2_lu.get(r['idx'], {}).get('faithfulness',   float('nan')),
    'context_recall'  : p2_lu.get(r['idx'], {}).get('context_recall', float('nan')),
    'avg_reranker_score': float(np.mean(r.get('reranker_scores', [0]))),
    'avg_bm25_score'    : float(np.mean(r.get('retrieval_scores', [0]))),
} for r in results_p1])

n, n_correct = len(df), int(df['is_correct'].sum())
label_acc  = n_correct / n
hallu_rate = 1 - label_acc
avg_f  = df['faithfulness'].mean()
avg_cr = df['context_recall'].mean()

print('=' * 60)
print(f'  CONTEXT RERANKING (BM25 top-{TOP_K_CANDIDATES} → CR top-{TOP_K_RETRIEVAL}) — {n} sampel')
print('=' * 60)
print(f'  {"Metrik":<32} {"Nilai":>10}')
print(f'  {"-"*42}')
print(f'  {"Label Accuracy":<32} {label_acc:>9.1%}')
print(f'  {"Hallucination Rate":<32} {hallu_rate:>9.1%}')
print(f'  {"-"*42}')
print(f'  {"Faithfulness (custom)":<32} {avg_f:>10.4f}')
print(f'  {"Context Recall (custom)":<32} {avg_cr:>10.4f}')
print(f'  {"Avg Reranker Score":<32} {df["avg_reranker_score"].mean():>10.4f}')
print(f'  {"Avg BM25 Score (top-5)":<32} {df["avg_bm25_score"].mean():>10.4f}')
print('=' * 60)
print('\nPer-label accuracy:')
for lbl in ['yes','no','maybe']:
    sub = df[df['ground_truth']==lbl]
    if len(sub):
        print(f'  {lbl:>5}: {sub["is_correct"].mean():.1%} ({int(sub["is_correct"].sum())}/{len(sub)}) '
              f'| faithfulness={sub["faithfulness"].mean():.3f} '
              f'| context_recall={sub["context_recall"].mean():.3f}')

  CONTEXT RERANKING (BM25 top-20 → CR top-5) — 500 sampel
  Metrik                                Nilai
  ------------------------------------------
  Label Accuracy                       68.4%
  Hallucination Rate                   31.6%
  ------------------------------------------
  Faithfulness (custom)                0.8980
  Context Recall (custom)              0.8203
  Avg Reranker Score                  -2.4110
  Avg BM25 Score (top-5)              24.0526

Per-label accuracy:
    yes: 86.5% (238/275) | faithfulness=0.887 | context_recall=0.841
     no: 63.5% (101/159) | faithfulness=0.927 | context_recall=0.805
  maybe: 4.5% (3/66) | faithfulness=0.872 | context_recall=0.769


---
## Perbandingan: Baseline vs QR vs Context Reranking

Jalankan setelah semua tiga notebook selesai.

In [13]:
configs_available = {
    'Baseline (BM25)'       : (BASELINE_PHASE1_PATH, BASELINE_PHASE2_CUSTOM_PATH),
    'BM25 + QR'             : (QR_PHASE1_PATH,       QR_PHASE2_CUSTOM_PATH),
    f'BM25 + CR (top-{TOP_K_CANDIDATES}→{TOP_K_RETRIEVAL})': (PHASE1_PATH, PHASE2_CUSTOM_PATH),
}

dfs = {}
for cfg_name, (p1_path, p2_path) in configs_available.items():
    if not p1_path.exists() or not p2_path.exists():
        print(f'[SKIP] {cfg_name}: file belum tersedia.')
        continue
    with open(p1_path) as f:
        p1 = json.load(f)['results']
    with open(p2_path) as f:
        p2 = json.load(f)['results']
    p2_lu = {r['idx']: r for r in p2}
    dfs[cfg_name] = pd.DataFrame([{
        'idx'            : r['idx'],
        'ground_truth'   : r['ground_truth'],
        'predicted_label': r['predicted_label'],
        'is_correct'     : r['is_correct'],
        'faithfulness'   : p2_lu.get(r['idx'], {}).get('faithfulness',   float('nan')),
        'context_recall' : p2_lu.get(r['idx'], {}).get('context_recall', float('nan')),
    } for r in p1])

if not dfs:
    print('Belum ada data untuk dibandingkan.')
else:
    print('=' * 80)
    print('PERBANDINGAN: BASELINE vs QR vs CONTEXT RERANKING')
    print('=' * 80)
    hdr = f'  {"Konfigurasi":<30} | {"Acc":>7} | {"Hallu":>6} | {"Faith":>7} | {"Ctx.R":>7}'
    print(hdr)
    print('  ' + '-' * 66)
    for cfg_name, dfc in dfs.items():
        acc  = dfc['is_correct'].mean()
        hr   = 1 - acc
        f_   = dfc['faithfulness'].mean()
        cr   = dfc['context_recall'].mean()
        print(f'  {cfg_name:<30} | {acc:>6.1%} | {hr:>5.1%} | {f_:>7.4f} | {cr:>7.4f}')
    print('=' * 80)

    if 'Baseline (BM25)' in dfs and f'BM25 + CR (top-{TOP_K_CANDIDATES}→{TOP_K_RETRIEVAL})' in dfs:
        bl  = dfs['Baseline (BM25)']
        cr_ = dfs[f'BM25 + CR (top-{TOP_K_CANDIDATES}→{TOP_K_RETRIEVAL})']
        print('\nDelta CR vs Baseline:')
        for metrik, bl_val, cr_val in [
            ('Label Accuracy',   bl['is_correct'].mean(),     cr_['is_correct'].mean()),
            ('Faithfulness',     bl['faithfulness'].mean(),   cr_['faithfulness'].mean()),
            ('Context Recall',   bl['context_recall'].mean(), cr_['context_recall'].mean()),
        ]:
            delta = cr_val - bl_val
            label = 'LEBIH BAIK ↑' if delta > 0.001 else ('LEBIH BURUK ↓' if delta < -0.001 else 'SAMA')
            print(f'  {metrik:<22}: {delta:+.4f}  ({label})')

    print('\nPer-label accuracy:')
    print(f'  {"Label":<7}' + ''.join(f' | {n:>28}' for n in dfs.keys()))
    print('  ' + '-' * (7 + 31 * len(dfs)))
    for lbl in ['yes', 'no', 'maybe']:
        row = f'  {lbl:<7}'
        for dfc in dfs.values():
            sub = dfc[dfc['ground_truth'] == lbl]
            if len(sub):
                row += f' | {sub["is_correct"].mean():>6.1%} (n={len(sub):>3})            '
            else:
                row += f' | {"N/A":>28}'
        print(row)

NameError: name 'BASELINE_PHASE2_CUSTOM_PATH' is not defined

---
## Analisis Kualitas Reranking

Inspeksi kualitatif: seberapa besar CrossEncoder mengubah urutan dari BM25?

In [14]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    cr_p1 = json.load(f)['results']

# Bandingkan: apakah reranker mengubah urutan?
# Simpan raw BM25 top-20 candidates untuk analisis
# (field 'retrieval_scores' di fase 1 adalah scores SETELAH rerank)
# Ukur reranker score spread: perbedaan antara dok terbaik dan terburuk dalam top-5

rr_spreads = []
for r in cr_p1:
    rr = r.get('reranker_scores', [])
    if len(rr) >= 2:
        rr_spreads.append(max(rr) - min(rr))

print('Analisis Reranker Score (top-5 per sampel):')
print('=' * 50)
all_rr = [s for r in cr_p1 for s in r.get('reranker_scores', [])]
print(f'  Mean score  : {np.mean(all_rr):.4f}')
print(f'  Std score   : {np.std(all_rr):.4f}')
print(f'  Min score   : {np.min(all_rr):.4f}')
print(f'  Max score   : {np.max(all_rr):.4f}')
if rr_spreads:
    print(f'  Avg spread (max-min per sampel): {np.mean(rr_spreads):.4f}')

print('\nContoh 10 sampel (reranker scores top-5):')
print('=' * 70)
for r in cr_p1[:10]:
    rr_scores = r.get('reranker_scores', [])
    bm_scores = r.get('retrieval_scores', [])
    status    = 'BENAR' if r['is_correct'] else 'SALAH'
    print(f'[{r["idx"]:3d}] GT={r["ground_truth"]} | Pred={r["predicted_label"]} | {status}')
    print(f'       Q: {r["question"][:70]}')
    if rr_scores:
        print(f'       Reranker: {[f"{s:.3f}" for s in rr_scores]}')
    if bm_scores:
        print(f'       BM25   : {[f"{s:.1f}" for s in bm_scores]}')
    print()

Analisis Reranker Score (top-5 per sampel):
  Mean score  : -2.4110
  Std score   : 5.8505
  Min score   : -11.3816
  Max score   : 10.8021
  Avg spread (max-min per sampel): 12.6850

Contoh 10 sampel (reranker scores top-5):
[  0] GT=yes | Pred=yes | BENAR
       Q: Do mitochondria play a role in remodelling lace plant leaves during pr
       Reranker: ['6.258', '-5.053', '-9.369', '-9.751', '-10.405']
       BM25   : ['51.2', '23.7', '15.9', '12.1', '13.3']

[  1] GT=no | Pred=yes | SALAH
       Q: Landolt C and snellen e acuity: differences in strabismus amblyopia?
       Reranker: ['8.947', '5.589', '2.035', '-3.391', '-10.751']
       BM25   : ['48.3', '38.4', '36.3', '21.1', '9.6']

[  2] GT=yes | Pred=yes | BENAR
       Q: Syncope during bathing in infants, a pediatric form of water-induced u
       Reranker: ['0.859', '-7.613', '-7.983', '-9.676', '-10.308']
       BM25   : ['24.7', '19.4', '12.7', '12.8', '14.5']

[  3] GT=no | Pred=maybe | SALAH
       Q: Are the long-term re